## Instructions

Before running this notebook, make sure the following files are located in the same directory:

- `Case2.ipynb`
- `inforce_policies_seriatim_0.5.0.csv`

Run the cells in `Case2.ipynb` to perform the analysis using the provided dataset.

In [ ]:
# Uncomment the following line to install required packages if needed:
# !pip install pandas numpy scikit-learn lightgbm optuna plotnine shap tqdm -q

In [ ]:
# Utility
from types import MethodType

# Data Manipulation
import pandas as pd
import numpy as np

# Predictive Modeling
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_poisson_deviance, mean_squared_error
from lightgbm import LGBMRegressor

# Optimization
import optuna

# Visualization
from plotnine import *

# Model Interpretation
import shap
from sklearn.cluster import KMeans

In [ ]:
class CONFIG:
    RANDOM_STATE = 0
    N_TRIALS_B = 0

# Analysis on Seriatim Dataset

In [ ]:
import os

# Define the file name
file_name = 'inforce_policies_seriatim_0.5.0.csv'

# Load the CSV file into data_df
if os.path.exists(file_name):
    data_df = pd.read_csv(file_name)
    print(f"Loaded {file_name} successfully.")
else:
    print(f"File not found: {file_name}")
    print("Please check if the file exists in the current working directory.")

In [ ]:
# Features
categorical_features = [
    'gender',
    'conversion_indicator',
    'renewal_status',
    'base_product',
    'corporate_indicator',
    'premium_waiver_rider',
    'advanced_medical_rider',
    'underwriting_method',
    'distribution_channel',
    'sales_prefecture_group'
]

numerical_features = [
    'attained_age',
    'duration_year',
    'sum_assured',
    'sum_assured_medical_comp_benefit_rider',
    'sum_assured_cancer_rider',
    'sum_assured_nursing_care_rider',
    'sum_assured_dementia_rider',
    'sum_assured_female_rider',
    'sum_assured_injury_rider',
    'number_of_riders',
]

# Convert Data Types

for col in categorical_features:
    if col in data_df.columns:
        data_df[col] = data_df[col].astype('category')

for col in numerical_features:
    if col in data_df.columns:
        data_df[col] = data_df[col].astype(float)

# Calculate Targets

data_df['p_index'] = data_df['actual_claim_amount'] / data_df['expected_claim_amount']

# assert data_df['p_index'].sum() == (data_df['claim_amount'] / data_df['expected_claim_amount']).sum()

data_df.head()

In [ ]:
# Split Data
train_df, test_df = train_test_split(
    data_df,
    test_size=0.2,
    random_state=CONFIG.RANDOM_STATE
)

all_features = numerical_features + categorical_features

X_train = train_df[all_features]
X_test  = test_df[all_features]

## Model Fitting

In [ ]:
def set_clipped_predict(model):
    if not hasattr(model, '_original_predict'):
        model._original_predict = model.predict
    def clipped_predict(self, X, **kwargs):
        preds = self._original_predict(X, **kwargs)
        return np.clip(preds, 1e-6, None)
    clipped_predict.__name__ = "predict"
    model.predict = MethodType(clipped_predict, model)
    return model

In [ ]:
# APPROACH : Squared error loss, direct modeling of the A/E ratio

y_train = train_df['p_index']
w_train = train_df['expected_claim_amount']

y_test  = test_df['p_index']
w_test  = test_df['expected_claim_amount']

# Hyperparameter tuning
def objective(trial):
    params = {
        "objective": "regression",
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 5),
        "min_child_samples": trial.suggest_int("min_child_samples", 100, 2000),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "random_state": 42,
        "verbose": -1
    }
    model = LGBMRegressor(**params)
    model.fit(
        X=X_train,
        y=y_train,
        sample_weight=w_train,
        categorical_feature=categorical_features
    )
    model = set_clipped_predict(model)
    preds = model.predict(X_test)
    loss = mean_squared_error(y_test, preds, sample_weight=w_test)
    return loss

params = {
    'learning_rate': 0.005774136286164429,
    'n_estimators': 615,
    'max_depth': 3,
    'min_child_samples': 1975,
    'reg_lambda': 0.005392079163994297,
    'reg_alpha': 1.2134394695880097
}

if CONFIG.N_TRIALS_B:
  study = optuna.create_study(direction="minimize")
  study.optimize(objective, n_trials=CONFIG.N_TRIALS_B)
  params.update(study.best_params)
  print("Best deviance:", study.best_value)
  print("Best parameters:", study.best_params)

# Build model_WLS using the tuned hyperparameters
model_WLS = LGBMRegressor(
    **params,
    objective='regression',
    random_state=42,
    verbose=-1
)
model_WLS.fit(
    X=X_train,
    y=y_train,
    sample_weight=w_train,
    categorical_feature=categorical_features
)
model_WLS = set_clipped_predict(model_WLS)

# Predict the A/E ratio
pred_index_WLS = model_WLS.predict(X_test)
print(f"Average A/E ratio: {np.average(pred_index_WLS, weights=test_df['expected_claim_amount']):.4f}")

## Model Evaluation

In [ ]:
# Baseline for the null model (assuming all policies have the overall average A/E ratio)
base_index = np.average(train_df['p_index'], weights=train_df['expected_claim_amount'])
assert train_df['actual_claim_amount'].sum() / train_df['expected_claim_amount'].sum() == base_index
null_pred_indices = np.full_like(test_df['p_index'], base_index)

# Calculate the Poisson deviance (target = actual A/E ratio, weight = expected claim amount)
null_deviance = mean_poisson_deviance(test_df['p_index'], null_pred_indices, sample_weight=test_df['expected_claim_amount'])
deviance_WLS = mean_poisson_deviance(
    test_df['p_index'],
    pred_index_WLS,
    sample_weight=test_df['expected_claim_amount']
)

print(f"Null Model Poisson Deviance  : {null_deviance:.4f}")
print(f"WLS Approach Poisson Deviance : {deviance_WLS:.4f} (Pseudo-R2: {1 - (deviance_WLS / null_deviance):.4%})")

## Interpretation

In [ ]:
model = model_WLS

In [ ]:
# Build the SHAP explainer and calculate SHAP values
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)

In [ ]:
FEATURE_DISPLAY_MAP={
    'gender': 'Gender',
    'sales_prefecture_group': 'Sales Prefecture',
    'distribution_channel': 'Distribution Channel',
    'underwriting_method': 'Underwriting Method',
    'renewal_status': 'Renewal Status',
    'rider_attachment_timing': 'Rider Attachment Timing',
    'conversion_indicator': 'Conversion Indicator',
    'corporate_indicator': 'Corporate Indicator',
    'attained_age': 'Attained Age',
    'duration_year': 'Duration Year',
    'sum_assured': 'Sum Assured',
    'sum_assured_medical_comp_benefit_rider': 'Sum Assured (Medical Comprehensive Benefit)',
    'sum_assured_cancer_rider': 'Sum Assured (Cancer)',
    'sum_assured_nursing_care_rider': 'Sum Assured (Nursing Care)',
    'sum_assured_dementia_rider': 'Sum Assured (Dementia)',
    'sum_assured_female_rider': 'Sum Assured (Female)',
    'sum_assured_injury_rider': 'Sum Assured (Injury)',
    'number_of_riders': 'Number of Riders',
    'base_product': 'Base Product',
    'premium_waiver_rider': 'Premium Waiver Rider',
    'advanced_medical_rider': 'Advanced Medical Rider'
}

In [ ]:
shap_values.feature_names = [FEATURE_DISPLAY_MAP[f] for f in shap_values.feature_names]

In [ ]:
pred_index = pred_index_WLS

n = min(4000, len(pred_index))
top_n_indices = np.argsort(-pred_index)[:n]

# Extract SHAP values for clustering (positive values only)
shap_matrix = shap_values.values[top_n_indices].clip(0)

# Run KMeans clustering
n_clusters = 12
kmeans = KMeans(n_clusters=n_clusters, random_state=CONFIG.RANDOM_STATE, n_init='auto')
clusters = kmeans.fit_predict(shap_matrix)

cluster_df = pd.DataFrame({
    'rank': range(1, n + 1),
    'test_df_index': top_n_indices,
    'predicted_index': pred_index[top_n_indices],
    'cluster': clusters
})

top_per_cluster = cluster_df.sort_values('rank').groupby('cluster').head(1).index.tolist()
top_labels = test_df.index[top_n_indices[top_per_cluster]]

top_df = test_df.loc[top_labels].copy()
top_df['predicted_index'] = pred_index[top_n_indices[top_per_cluster]]
top_df['assigned_cluster'] = cluster_df.loc[top_per_cluster, 'cluster'].values

print("--- Top Record for Each Cluster ---")
display(top_df[['assigned_cluster', 'predicted_index'] + all_features])

In [ ]:
# Generate waterfall plots for each record
for i, idx in enumerate(top_n_indices[top_per_cluster]):
    print(f"\n--- [Waterfall Plot {i+1}] Index: {idx}, Predicted Index: {pred_index[idx]:.4f} ---\n")
    shap.plots.waterfall(shap_values[idx], max_display=8)